# W4111 — classicmodels API Demo Notebook (Task 6)

This notebook tests and demonstrates all implemented tasks:
- **Task 1**: `MySQLDataService` — direct method calls
- **Task 2**: `CustomerResource` — direct calls + HTTP
- **Task 3**: `OrderResource` — direct calls + HTTP
- **Task 4**: `OrderDetailsResource` — direct calls + HTTP
- **Task 5**: HTTP routes via `httpx`

**Prerequisites**: The FastAPI server must be running:
```
MYSQL_UNIX_SOCKET=/tmp/mysql_custom.sock python -m app.main
```
and the `classicmodels` database must be loaded.

In [1]:
import sys, os
# Make sure the project root is on the path
project_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Set env variables so the data services can connect to MySQL
os.environ.setdefault('MYSQL_HOST', 'localhost')
os.environ.setdefault('MYSQL_PORT', '3307')
os.environ.setdefault('MYSQL_USER', 'root')
os.environ.setdefault('MYSQL_PASSWORD', '')
os.environ.setdefault('MYSQL_DATABASE', 'classicmodels')
os.environ.setdefault('MYSQL_UNIX_SOCKET', '/tmp/mysql_custom.sock')

BASE_URL = 'http://localhost:8000'
print('Environment configured, project root:', project_root)

Environment configured, project root: /home/sun/hermes/W4111-Template_Web_Application


## Task 1 — MySQLDataService

In [2]:
from app.services.MySQLDataService import MySQLDataService

# Instantiate a MySQLDataService for the customers table
svc = MySQLDataService({
    'table': 'customers',
    'primary_key_field': 'customerNumber',
})

# retrieveByPrimaryKey
cust = svc.retrieveByPrimaryKey(103)
print('retrieveByPrimaryKey(103):', cust)

retrieveByPrimaryKey(103): {'customerNumber': 103, 'customerName': 'Atelier graphique', 'contactLastName': 'Schmitt', 'contactFirstName': 'Carine ', 'phone': '40.32.2555', 'addressLine1': '54, rue Royale', 'addressLine2': None, 'city': 'Nantes', 'state': None, 'postalCode': '44000', 'country': 'France', 'salesRepEmployeeNumber': 1370, 'creditLimit': Decimal('21000.00')}


In [3]:
# retrieveByTemplate — find all customers in France
french = svc.retrieveByTemplate({'country': 'France'})
print(f'French customers ({len(french)}):')
for c in french[:3]:
    print(' ', c['customerNumber'], c['customerName'], c['city'])

French customers (12):
  103 Atelier graphique Nantes
  119 La Rochelle Gifts Nantes
  146 Saveley & Henriot, Co. Lyon


In [4]:
# create, updateByPrimaryKey, deleteByPrimaryKey
new_pk = svc.create({
    'customerNumber': 99999,
    'customerName': 'Test Corp',
    'contactLastName': 'Doe',
    'contactFirstName': 'John',
    'phone': '555-0000',
    'addressLine1': '1 Test St',
    'city': 'Testville',
    'country': 'USA',
})
print('Created customerNumber:', new_pk)

updated = svc.updateByPrimaryKey(99999, {'city': 'NewTestville'})
print('Rows updated:', updated)
print('After update:', svc.retrieveByPrimaryKey(99999).get('city'))

deleted = svc.deleteByPrimaryKey(99999)
print('Rows deleted:', deleted)
print('After delete:', svc.retrieveByPrimaryKey(99999))

Created customerNumber: 99999
Rows updated: 1
After update: NewTestville
Rows deleted: 1
After delete: {}


## Task 2 — CustomerResource (direct)

In [5]:
from app.resources.CustomerResource import Customer, CustomerResource

cr = CustomerResource()

# get (list all US customers)
us_customers = cr.get({'country': 'USA'})
print(f'US customers: {len(us_customers.items)}')
print('First:', us_customers.items[0])

US customers: 36
First: customerNumber=112 customerName='Signal Gift Stores' contactLastName='King' contactFirstName='Jean' phone='7025551838' addressLine1='8489 Strong St.' addressLine2=None city='Las Vegas' state='NV' postalCode='83030' country='USA' salesRepEmployeeNumber=1166 creditLimit=Decimal('71800.00')


In [6]:
# get_by_id
c = cr.get_by_id('103')
print('get_by_id(103):', c)

get_by_id(103): customerNumber=103 customerName='Atelier graphique' contactLastName='Schmitt' contactFirstName='Carine ' phone='40.32.2555' addressLine1='54, rue Royale' addressLine2=None city='Nantes' state=None postalCode='44000' country='France' salesRepEmployeeNumber=1370 creditLimit=Decimal('21000.00')


In [7]:
# post, put, delete round-trip
new_customer = Customer(
    customerNumber=88888,
    customerName='Notebook Test Ltd',
    contactLastName='Smith',
    contactFirstName='Alice',
    phone='555-1234',
    addressLine1='42 Notebook Ave',
    city='Bookville',
    country='Canada',
)
pk = cr.post(new_customer)
print('Created pk:', pk)

rows = cr.put('88888', Customer(customerName='Notebook Test Ltd Updated', contactLastName='Smith',
              contactFirstName='Alice', phone='555-9999', addressLine1='42 Notebook Ave',
              city='Bookville', country='Canada'))
print('Updated rows:', rows)
print('After update:', cr.get_by_id('88888').customerName)

d = cr.delete('88888')
print('Deleted:', d)

Created pk: 88888
Updated rows: 1
After update: Notebook Test Ltd Updated


Deleted: 1


## Task 3 — OrderResource (direct)

In [8]:
from app.resources.OrderResource import Order, OrderResource

orr = OrderResource()

# get all shipped orders
shipped = orr.get({'status': 'Shipped'})
print(f'Shipped orders: {len(shipped.items)}')

# get_by_id
o = orr.get_by_id('10100')
print('Order 10100:', o)

Shipped orders: 303
Order 10100: orderNumber=10100 orderDate=datetime.date(2003, 1, 6) requiredDate=datetime.date(2003, 1, 13) shippedDate=datetime.date(2003, 1, 10) status='Shipped' comments=None customerNumber=363


In [9]:
from datetime import date

# post, put, delete
new_order = Order(
    orderNumber=99998,
    orderDate=date(2024, 1, 1),
    requiredDate=date(2024, 1, 15),
    status='In Process',
    customerNumber=103,
)
pk = orr.post(new_order)
print('Created orderNumber:', pk)

rows = orr.put('99998', Order(status='Shipped', orderDate=date(2024,1,1),
               requiredDate=date(2024,1,15), customerNumber=103))
print('Updated rows:', rows)
print('After update status:', orr.get_by_id('99998').status)

d = orr.delete('99998')
print('Deleted:', d)

Created orderNumber: 99998


Updated rows: 1
After update status: Shipped


Deleted: 1


## Task 4 — OrderDetailsResource (direct)

In [10]:
from app.resources.OrderDetailsResource import OrderDetail, OrderDetailsResource

odr = OrderDetailsResource()

# get all order details for order 10100
details = odr.get({'orderNumber': 10100})
print(f'Order details for 10100 ({len(details.items)} items):')
for d in details.items:
    print(f'  {d.productCode}  qty={d.quantityOrdered}  price={d.priceEach}')

Order details for 10100 (4 items):
  S18_1749  qty=30  price=136.00
  S18_2248  qty=50  price=55.09
  S18_4409  qty=22  price=75.46
  S24_3969  qty=49  price=35.29


In [11]:
# get_by_id with composite key
detail = odr.get_by_id((10100, 'S18_1749'))
print('get_by_id((10100, S18_1749)):', detail)

get_by_id((10100, S18_1749)): orderNumber=10100 productCode='S18_1749' quantityOrdered=30 priceEach=Decimal('136.00') orderLineNumber=3


## Task 5 — HTTP API via httpx

In [12]:
import httpx, json

client = httpx.Client(base_url=BASE_URL)

# Health check
r = client.get('/health')
print('GET /health:', r.status_code, r.json())

GET /health: 200 {'status': 'ok'}


In [13]:
# GET /customers — list all customers
r = client.get('/customers')
data = r.json()
print(f'GET /customers: {r.status_code}, total items: {len(data["items"])}')
print('First customer:', data['items'][0])

GET /customers: 200, total items: 122


First customer: {'customerNumber': 103, 'customerName': 'Atelier graphique', 'contactLastName': 'Schmitt', 'contactFirstName': 'Carine ', 'phone': '40.32.2555', 'addressLine1': '54, rue Royale', 'addressLine2': None, 'city': 'Nantes', 'state': None, 'postalCode': '44000', 'country': 'France', 'salesRepEmployeeNumber': 1370, 'creditLimit': '21000.00'}


In [14]:
# GET /customers?country=France
r = client.get('/customers', params={'country': 'France'})
print(f'GET /customers?country=France: {r.status_code}, count: {len(r.json()["items"])}')

GET /customers?country=France: 200, count: 12


In [15]:
# GET /customers/{customerNumber}
r = client.get('/customers/103')
print(f'GET /customers/103: {r.status_code}')
print(r.json())

GET /customers/103: 200
{'customerNumber': 103, 'customerName': 'Atelier graphique', 'contactLastName': 'Schmitt', 'contactFirstName': 'Carine ', 'phone': '40.32.2555', 'addressLine1': '54, rue Royale', 'addressLine2': None, 'city': 'Nantes', 'state': None, 'postalCode': '44000', 'country': 'France', 'salesRepEmployeeNumber': 1370, 'creditLimit': '21000.00'}


In [16]:
# GET /customers/99999 — should 404
r = client.get('/customers/99999')
print(f'GET /customers/99999: {r.status_code} (expected 404)')

GET /customers/99999: 404 (expected 404)


In [17]:
# POST /customers — create a new customer
new_cust = {
    'customerNumber': 77777,
    'customerName': 'HTTP Test Inc',
    'contactLastName': 'HTTP',
    'contactFirstName': 'Test',
    'phone': '555-7777',
    'addressLine1': '7 API Lane',
    'city': 'APItown',
    'country': 'USA',
}
r = client.post('/customers', json=new_cust)
print(f'POST /customers: {r.status_code}, pk returned: {r.json()}')

POST /customers: 200, pk returned: 77777


In [18]:
# PUT /customers/77777 — update city
r = client.put('/customers/77777', json={
    'customerName': 'HTTP Test Inc Updated',
    'contactLastName': 'HTTP',
    'contactFirstName': 'Test',
    'phone': '555-7777',
    'addressLine1': '7 API Lane',
    'city': 'NewAPItown',
    'country': 'USA',
})
print(f'PUT /customers/77777: {r.status_code}', r.json())
r2 = client.get('/customers/77777')
print('After PUT:', r2.json()['city'])

PUT /customers/77777: 200 {'updated': 1}
After PUT: NewAPItown


In [19]:
# DELETE /customers/77777
r = client.delete('/customers/77777')
print(f'DELETE /customers/77777: {r.status_code}', r.json())
r2 = client.get('/customers/77777')
print(f'After DELETE, GET /customers/77777: {r2.status_code} (expected 404)')

DELETE /customers/77777: 200 {'deleted': 1}
After DELETE, GET /customers/77777: 404 (expected 404)


In [20]:
# GET /orders — list all orders
r = client.get('/orders')
print(f'GET /orders: {r.status_code}, total: {len(r.json()["items"])}')

GET /orders: 200, total: 326


In [21]:
# GET /orders?status=Shipped
r = client.get('/orders', params={'status': 'Shipped'})
print(f'GET /orders?status=Shipped: {r.status_code}, count: {len(r.json()["items"])}')

GET /orders?status=Shipped: 200, count: 303


In [22]:
# GET /orders/10100
r = client.get('/orders/10100')
print(f'GET /orders/10100: {r.status_code}')
print(r.json())

GET /orders/10100: 200
{'orderNumber': 10100, 'orderDate': '2003-01-06', 'requiredDate': '2003-01-13', 'shippedDate': '2003-01-10', 'status': 'Shipped', 'comments': None, 'customerNumber': 363}


In [23]:
# GET /orders/10100/orderdetails
r = client.get('/orders/10100/orderdetails')
items = r.json()['items']
print(f'GET /orders/10100/orderdetails: {r.status_code}, {len(items)} line items')
for item in items:
    print(f'  {item["productCode"]}  qty={item["quantityOrdered"]}  price={item["priceEach"]}')

GET /orders/10100/orderdetails: 200, 4 line items
  S18_1749  qty=30  price=136.00
  S18_2248  qty=50  price=55.09
  S18_4409  qty=22  price=75.46
  S24_3969  qty=49  price=35.29


In [24]:
# GET /orders/10100/orderdetails/S18_1749 — single composite-key row
r = client.get('/orders/10100/orderdetails/S18_1749')
print(f'GET /orders/10100/orderdetails/S18_1749: {r.status_code}')
print(r.json())

GET /orders/10100/orderdetails/S18_1749: 200
{'orderNumber': 10100, 'productCode': 'S18_1749', 'quantityOrdered': 30, 'priceEach': '136.00', 'orderLineNumber': 3}


In [25]:
# GET /orderdetails — full collection
r = client.get('/orderdetails')
print(f'GET /orderdetails: {r.status_code}, total line items: {len(r.json()["items"])}')

GET /orderdetails: 200, total line items: 2996


In [26]:
print('All tests complete!')
client.close()

All tests complete!
